In [ ]:
from typing import List
import math
import random
import tqdm

In [31]:
Vector = List[float]

def dot(v: Vector, w: Vector) -> float:
    """Calcula v_1 * w_1 + ... + v_n * w_n"""
    assert len(v) == len(w), "Los vectores deben tener la misma longitud"
    
    return sum(v_i * w_i for v_i, w_i in zip(v, w))

def gradient_step(v: Vector, gradient: Vector, step_size: float) -> Vector:
    """
    Se mueve step_size en la dirección de gradient desde v.
    En el entrenamiento, usamos -learning_rate para ir 'cuesta abajo'.
    """
    assert len(v) == len(gradient)
    step = [step_size * g_i for g_i in gradient]
    return [v_i + s_i for v_i, s_i in zip(v, step)]

def squared_distance(v: Vector, w: Vector) -> float:
    """Calcula (v_1 - w_1)**2 + ... + (v_n - w_n)**2"""
    return sum((v_i - w_i) ** 2 for v_i, w_i in zip(v, w))

In [3]:
def step_function(x: float) -> float:
    return 1.0 if x>= 0 else 0.0

def perceptron_output(weights: Vector, bias: float, x: Vector) -> float:
    """devuelve 1 si el perctron se 'activa', 0 si no"""
    calculation = dot(weights, x) + bias
    return step_function(calculation)

In [4]:
and_weights = [2., 2]
and_bias = -3.
assert perceptron_output(and_weights, and_bias, [1, 1]) == 1
assert perceptron_output(and_weights, and_bias, [0, 1]) == 0
assert perceptron_output(and_weights, and_bias, [1, 0]) == 0
assert perceptron_output(and_weights, and_bias, [0, 0]) == 0

In [5]:
or_weights = [2., 2]
or_bias = -1.
assert perceptron_output(or_weights, or_bias, [1, 1]) == 1
assert perceptron_output(or_weights, or_bias, [0, 1]) == 1
assert perceptron_output(or_weights, or_bias, [1, 0]) == 1
assert perceptron_output(or_weights, or_bias, [0, 0]) == 0

In [6]:
or_weights = [2., 2]
or_bias = -1.
assert perceptron_output(or_weights, or_bias, [1, 1]) == 1
assert perceptron_output(or_weights, or_bias, [0, 1]) == 1
assert perceptron_output(or_weights, or_bias, [1, 0]) == 1
assert perceptron_output(or_weights, or_bias, [0, 0]) == 0

In [7]:
not_weights = [-2.]
not_bias = 1.

assert perceptron_output(not_weights, not_bias, [0]) == 1
assert perceptron_output(not_weights, not_bias, [1]) == 0

In [8]:
# 1. Compuerta AND (retorna el valor mínimo entre las entradas)
and_gate = min

# 2. Compuerta OR (retorna el valor máximo entre las entradas)
or_gate = max

# 3. Compuerta XOR (Solución con 'def' en lugar de 'lambda')
def xor_gate(x: float, y: float) -> int:
    """Retorna 1 si las entradas son diferentes, 0 si son iguales"""
    if x == y:
        return 0
    else:
        return 1

In [9]:
def sigmoid(t: float) -> float:
    return 1 / ( 1 + math.exp(-t))

In [10]:
def neuron_output(weights: Vector, inputs: Vector) -> float:
    # weights incluye el termino de sesgo, inputs incluye un 1
    return sigmoid(dot(weights, inputs))

In [11]:
def feed_forward(neural_networrk: List[List[Vector]], input_vector: Vector) -> List[Vector]:
    """Alimenta el vector de entrada a través de la red neuronal. Devuelve las salidas de todas las capas (no solo la última)"""
    outputs: List[Vector] = []
    for layer in neural_networrk:
        input_with_bias = input_vector + [1]
        output = [neuron_output(neuron, input_with_bias) for neuron in layer]
        outputs.append(output)

        # lueog la entrada de la capa siguiente es la salida de esta
        input_vector = output
    return outputs

In [12]:
xor_network = [# hidden layer
               [[20., 20, -30],      # 'and' neuron
                [20., 20, -10]],     # 'or'  neuron
               # output layer
               [[-60., 60, -30]]]    # '2nd input but not 1st input' neuron

# feed_forward returns the outputs of all layers, so the [-1] gets the
# final output, and the [0] gets the value out of the resulting vector
assert 0.000 < feed_forward(xor_network, [0, 0])[-1][0] < 0.001
assert 0.999 < feed_forward(xor_network, [1, 0])[-1][0] < 1.000
assert 0.999 < feed_forward(xor_network, [0, 1])[-1][0] < 1.000
assert 0.000 < feed_forward(xor_network, [1, 1])[-1][0] < 0.001

In [13]:
def sqerror_gradients(network: List[List[Vector]], input_vector: Vector, target_vector: Vector) -> List[List[Vector]]:
    """Dada una red neuronal, un vector de entrada y un vector objetivo, Hacer una predicción y calcular el gradiente del error al cuadrado Pérdida con respecto a los pesos de las neuronas."""
    # paso adelante
    hidden_outputs, outputs = feed_forward(network, input_vector)
    # gradientes con respecto a obtener salidas de preactivacion de neurona
    output_deltas = [output * (1 - output) * (output - target) for output, target in zip(outputs, target_vector)]
    #gradientes con respecto a obtener salidas de pesos de neurona
    output_grads = [[output_deltas[i] * hidden_output for hidden_output in hidden_outputs + [1]] for i, output_neuron in enumerate(network[-1])]
    # gradientes cpn respecto a salidas ocultas de preactivacion de neurona
    hidden_deltas = [hidden_output * (1-hidden_output)* dot(output_deltas, [n[i] for n in network[-1]]) for i, hidden_output in enumerate(hidden_outputs)]
    # gradientes con respecto a salidas ocultas de pesos de naurona
    hidden_grads = [[hidden_deltas[i] * input for input in input_vector + [1]] for i, hidden_neuron in enumerate(network[0])]

    return [hidden_grads, output_grads]

In [15]:
random.seed(0)
# datos de entrenamiento
xs = [[0., 0], [0., 1], [1., 0], [1., 1]]
ys = [[0.], [1.], [1.], [0.]]
# empieza con pesos aleatorios
network = [ # hidden layer: 2 inputs -> 2 outputs
                [[random.random() for _ in range(2 + 1)],   # 1st hidden neuron
                 [random.random() for _ in range(2 + 1)]],  # 2nd hidden neuron
                # output layer: 2 inputs -> 1 output
                [[random.random() for _ in range(2 + 1)]]   # 1st output neuron
]

In [20]:
learning_rate = 1.0
for epoch in tqdm.trange(20000, desc="neural net for xor"):
    for x, y in zip(xs, ys):
        gradients = sqerror_gradients(network, x, y)
        # da un paso de gradiente para cada neurona en cada capa
        network = [[gradient_step(neuron, grad, -learning_rate) for neuron, grad in zip(layer, layer_grad)] for layer, layer_grad in zip(network, gradients)]
# comprueba que descubrio XOR
assert feed_forward(network, [0, 0])[-1][0] < 0.01
assert feed_forward(network, [0, 1])[-1][0] > 0.99
assert feed_forward(network, [1, 0])[-1][0] > 0.99
assert feed_forward(network, [1, 1])[-1][0] < 0.01

neural net for xor: 100%|██████████| 20000/20000 [00:00<00:00, 22110.20it/s]


In [24]:
def fizz_buzz_encode(x: int) -> Vector:
    if x % 15 == 0:
        return [0, 0, 0, 1]
    elif x % 5 == 0:
        return [0, 0, 1, 0]
    elif x % 3 == 0:
        return [0, 1, 0, 0]
    else:
        return [1, 0, 0, 0]
    
assert fizz_buzz_encode(2) == [1, 0, 0, 0]
assert fizz_buzz_encode(6) == [0, 1, 0, 0]
assert fizz_buzz_encode(10) == [0, 0, 1, 0]
assert fizz_buzz_encode(30) == [0, 0, 0, 1]

In [26]:
def binary_encode(x: int) -> Vector:
    binary: List[float] = []
    for i in range(10):
        binary.append(x % 2)
        x = x // 2
    return binary

assert binary_encode(0) == [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
assert binary_encode(1) == [1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
assert binary_encode(10) == [0, 1, 0, 1, 0, 0, 0, 0, 0, 0]
assert binary_encode(101) == [1, 0, 1, 0, 0, 1, 1, 0, 0, 0]
assert binary_encode(999) == [1, 1, 1, 0, 0, 1, 1, 1, 1, 1]

In [27]:
xs = [binary_encode(n) for n in range(101, 1024)]
ys = [fizz_buzz_encode(n) for n in range(101, 1024)]

In [29]:
NUM_HIDDEN = 25
network = [
    # capa oculta : 10 entradas -> NUM_HIDDEN salidas
    [[random.random() for _ in range (10 + 1)] for _ in range(NUM_HIDDEN)],
    # output_layer: NUM_HIDDEN entradas -> 4 salidas
    [[random.random() for _ in range(NUM_HIDDEN + 1)] for _ in range(4)]
]

In [32]:
learning_rate = 1.0
with tqdm.trange(500) as t:
    for epoch in t:
        epoch_loss = 0.0
        for x, y in zip(xs, ys):
            predicted = feed_forward(network, x)[-1]
            epoch_loss += squared_distance(predicted, y)
            gradients = sqerror_gradients(network, x, y)
            # da un paso de gradiente para cada neurona en cada capa
            network = [[gradient_step(neuron, grad, -learning_rate) for neuron, grad in zip(layer, layer_grad)] for layer, layer_grad in zip(network, gradients)]
    t.set_description(f"fizz buzz (loss: {epoch_loss:.2f})")

100%|██████████| 500/500 [01:41<00:00,  4.91it/s]


In [35]:
def argmax(xs: list) -> int:
    """Devuelve el índice del mayor valor"""
    return max(range(len(xs)), key=lambda i: xs[i])

assert argmax([0, -1]) == 0                 # items[0] es el más grande
assert argmax([-1, 0]) == 1                 # items[1] es el más grande
assert argmax([-1, 10, 5, 20, -3]) == 3     # items[3] es el mas grande

In [38]:
# 1. IMPORTANTE: Inicializar el contador
num_correct = 0

for n in range(1, 101):
    x = binary_encode(n)
    
    # Obtenemos la predicción de la red (el índice con mayor probabilidad)
    predicted = argmax(feed_forward(network, x)[-1])
    
    # Obtenemos el índice real basado en las reglas de FizzBuzz
    actual = argmax(fizz_buzz_encode(n))
    
    labels = [str(n), "fizz", "buzz", "fizzbuzz"]
    
    # Imprimimos: Número | Predicción | Real
    print(f"{n:3} | Predicción: {labels[predicted]:10} | Real: {labels[actual]:10}")
    
    if predicted == actual:
        num_correct += 1

  1 | Predicción: 1          | Real: 1         
  2 | Predicción: 2          | Real: 2         
  3 | Predicción: fizz       | Real: fizz      
  4 | Predicción: buzz       | Real: 4         
  5 | Predicción: buzz       | Real: buzz      
  6 | Predicción: fizz       | Real: fizz      
  7 | Predicción: 7          | Real: 7         
  8 | Predicción: 8          | Real: 8         
  9 | Predicción: fizz       | Real: fizz      
 10 | Predicción: buzz       | Real: buzz      
 11 | Predicción: 11         | Real: 11        
 12 | Predicción: fizz       | Real: fizz      
 13 | Predicción: 13         | Real: 13        
 14 | Predicción: 14         | Real: 14        
 15 | Predicción: fizzbuzz   | Real: fizzbuzz  
 16 | Predicción: 16         | Real: 16        
 17 | Predicción: 17         | Real: 17        
 18 | Predicción: fizz       | Real: fizz      
 19 | Predicción: 19         | Real: 19        
 20 | Predicción: buzz       | Real: buzz      
 21 | Predicción: fizz       | Real: fiz